# CH05 -- Notebook thực hành
Chạy lần lượt từng ô từ trên xuống. Đặt notebook ở thư mục gốc dự án (cùng cấp với thư mục `data/` và `figures/`).

### Ô lệnh 1 -- Nạp thư viện và dữ liệu

In [ ]:
import sys
sys.path.append("code")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from common import credit_split, xgb_clf

pd.set_option("display.width", 80)
pd.set_option("display.precision", 3)
SEED = 42

X_train, X_test, y_train, y_test = credit_split()
n0, n1 = (y_train == 0).sum(), (y_train == 1).sum()
print(f"Tập train: {n0} không vỡ nợ, {n1} vỡ nợ, tỷ số n0/n1 = {n0 / n1:.2f}")


### Ô lệnh 2 -- Hàm mất mát log loss, có và không có trọng số, tính tay cho 4 khách hàng

In [ ]:
y_true = np.array([1, 1, 0, 0])
p_hat = np.array([0.30, 0.80, 0.20, 0.60])     # xác suất vỡ nợ mà mô hình dự đoán
w = 3.5                                        # trọng số cho lớp vỡ nợ (lớp 1)

loss = -(y_true * np.log(p_hat) + (1 - y_true) * np.log(1 - p_hat))   # mất mát từng người
weight = np.where(y_true == 1, w, 1.0)
print(pd.DataFrame({"y": y_true, "p_hat": p_hat, "mất mát": loss.round(3),
                    "trọng số": weight, "mất mát x trọng số": (loss * weight).round(3)}))
print(f"\nTổng mất mát -- không trọng số: {loss.sum():.3f} | có trọng số: {(loss * weight).sum():.3f}")


### Ô lệnh 3 -- Kiểm chứng: trọng số 3 cho một mẫu = nhân bản mẫu đó thành 3 bản

In [ ]:
Xs = StandardScaler().fit_transform(X_train)[:2000]      # lấy 2 000 dòng cho nhanh
ys = y_train.values[:2000]

# Cách A: trọng số 3 cho mọi khách hàng vỡ nợ
lr_a = LogisticRegression(max_iter=5000).fit(Xs, ys, sample_weight=np.where(ys == 1, 3.0, 1.0))
# Cách B: chép mỗi khách hàng vỡ nợ thành 3 bản, không dùng trọng số
X_dup = np.vstack([Xs, Xs[ys == 1], Xs[ys == 1]])
y_dup = np.concatenate([ys, ys[ys == 1], ys[ys == 1]])
lr_b = LogisticRegression(max_iter=5000).fit(X_dup, y_dup)

print("5 hệ số đầu -- cách A:", lr_a.coef_[0][:5].round(4))
print("5 hệ số đầu -- cách B:", lr_b.coef_[0][:5].round(4))
print("Chênh lệch lớn nhất giữa hai cách:", f"{np.abs(lr_a.coef_ - lr_b.coef_).max():.1e}")


### Ô lệnh 4 -- Trọng số "balanced" của scikit-learn

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

w_bal = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
ratio = w_bal[1] / w_bal[0]
print(f"w0 = {w_bal[0]:.3f}, w1 = {w_bal[1]:.3f}, tỷ số w1/w0 = {ratio:.3f}")
print(f"Tổng trọng số lớp 0: {w_bal[0] * n0:.0f} | lớp 1: {w_bal[1] * n1:.0f}")


### Ô lệnh 5 -- Quét trọng số lớp dương w cho XGBoost (đánh giá chéo, ~40 giây)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision", "F1": "f1",
           "Recall": "recall", "Precision": "precision"}
rows = []
for w_ in [1, 1.5, 2, 3, round(ratio, 2), 5, 8]:
    r = cross_validate(xgb_clf(scale_pos_weight=w_), X_train, y_train, cv=skf, scoring=scoring)
    rows.append({"w": w_, **{k: r["test_" + k].mean() for k in scoring}})
sweep = pd.DataFrame(rows).set_index("w")
print(sweep)

ax = sweep[["PR-AUC", "F1", "Recall", "Precision"]].plot(marker="o", figsize=(5.2, 2.9))
ax.axvline(ratio, ls="--", color="gray")
ax.set_xlabel("Trọng số lớp dương w (scale_pos_weight)"); ax.set_ylabel("Điểm CV trung bình")
plt.savefig("figures/ch05_s05_sweep.pdf", bbox_inches="tight")


### Ô lệnh 6 -- Trọng số w "uốn" thang xác suất theo công thức p_w = w p / (w p + 1 - p)

In [ ]:
p = np.array([0.05, 0.10, 0.221, 0.30, 0.50, 0.80])
w_ = ratio
p_w = w_ * p / (w_ * p + 1 - p)
print(pd.DataFrame({"p (không trọng số)": p, f"p_w (w = {w_:.2f})": p_w.round(3)}).to_string(index=False))
print(f"\nNgưỡng tương đương: p_w >= 0,5  <=>  p >= 1/(1+w) = {1 / (1 + w_):.3f}")


### Ô lệnh 7 -- Kiểm chứng trên dữ liệu thật: trọng số w và ngưỡng 1/(1+w) cho cùng quyết định?

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, average_precision_score
from scipy.stats import spearmanr

p1 = xgb_clf().fit(X_train, y_train).predict_proba(X_test)[:, 1]                       # w = 1
pw = xgb_clf(scale_pos_weight=ratio).fit(X_train, y_train).predict_proba(X_test)[:, 1]  # w = 3,52
t_eq = 1 / (1 + ratio)

def report(p, t):
    pred = (p >= t).astype(int)
    return {"F1": f1_score(y_test, pred), "Recall": recall_score(y_test, pred),
            "Precision": precision_score(y_test, pred), "PR-AUC": average_precision_score(y_test, p)}

print(pd.DataFrame({"w = 1, t = 0,5": report(p1, 0.5),
                    f"w = {ratio:.2f}, t = 0,5": report(pw, 0.5),
                    f"w = 1, t = {t_eq:.3f}": report(p1, t_eq)}).T.round(3))
print(f"\nTương quan hạng Spearman giữa hai bộ xác suất: {spearmanr(p1, pw).correlation:.4f}")


### Ô lệnh 8 -- Vẽ xác suất của mô hình có trọng số theo mô hình không trọng số

In [ ]:
grid = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(4.2, 3.4))
ax.scatter(p1, pw, s=3, alpha=0.3, color="#2a78d6", label="6 000 khách hàng test")
ax.plot(grid, ratio * grid / (ratio * grid + 1 - grid), color="#eb6834", lw=2, label="công thức lý thuyết")
ax.plot([0, 1], [0, 1], ls=":", color="gray")
ax.set_xlabel("p (w = 1)"); ax.set_ylabel(f"p_w (w = {ratio:.2f})"); ax.legend(fontsize=7)
plt.savefig("figures/ch05_s08_pw.pdf", bbox_inches="tight")


### Ô lệnh 9 -- Cái giá phải trả: xác suất bị méo

In [ ]:
print(f"Tỷ lệ vỡ nợ thật trên tập test : {y_test.mean():.3f}")
print(f"Xác suất TB -- w = 1           : {p1.mean():.3f}")
print(f"Xác suất TB -- w = {ratio:.2f}        : {pw.mean():.3f}")

# Khử trọng số: đảo ngược công thức ở Ô lệnh 6
p_back = pw / (pw + ratio * (1 - pw))
print(f"Xác suất TB sau khi khử trọng số: {p_back.mean():.3f}")


### Ô lệnh 10 -- Chọn ngưỡng từ chi phí: t* = 1 / (1 + C_FN / C_FP)

In [ ]:
def total_cost(p, t, c_fn, c_fp=1):
    pred = (p >= t).astype(int)
    fn = int(((pred == 0) & (y_test == 1)).sum())    # bỏ sót người vỡ nợ
    fp = int(((pred == 1) & (y_test == 0)).sum())    # báo động nhầm
    return fn, fp, fn * c_fn + fp * c_fp

rows = []
for c in [1, 3, 5, 10]:
    t_star = 1 / (1 + c)
    fn, fp, cost = total_cost(p1, t_star, c)
    rows.append({"C_FN/C_FP": c, "t*": round(t_star, 3), "FN": fn, "FP": fp,
                 "Chi phí (t*)": cost, "Chi phí (t = 0,5)": total_cost(p1, 0.5, c)[2]})
print(pd.DataFrame(rows).to_string(index=False))


### Ô lệnh 11 -- Vẽ tổng chi phí theo ngưỡng (C_FN/C_FP = 5)

In [ ]:
ts = np.linspace(0.02, 0.9, 89)
costs = [total_cost(p1, t, 5)[2] for t in ts]
print(f"Ngưỡng có chi phí thấp nhất trên tập test: {ts[np.argmin(costs)]:.2f}"
      f" (lý thuyết: {1 / 6:.3f})")

fig, ax = plt.subplots(figsize=(5, 2.7))
ax.plot(ts, costs)
ax.axvline(1 / 6, ls="--", color="gray"); ax.axvline(0.5, ls=":", color="gray")
ax.set_xlabel("Ngưỡng t"); ax.set_ylabel("Tổng chi phí (C_FP = 1)")
plt.savefig("figures/ch05_s11_cost.pdf", bbox_inches="tight")


### Ô lệnh 12 -- Với hồi quy logistic: class_weight cũng chỉ dịch chuyển sự đánh đổi

In [ ]:
rows = []
for w_ in [1, 2, 3, round(ratio, 2), 5]:
    lr = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=2000, class_weight={0: 1, 1: w_}))
    r = cross_validate(lr, X_train, y_train, cv=skf, scoring=scoring)
    rows.append({"w": w_, **{k: r["test_" + k].mean() for k in scoring}})
print(pd.DataFrame(rows).set_index("w"))
